# 数据清洗过程

## 一. 清洗 panel_data.xlsx 中的数据。清洗过程包括：
- 数据读取
- 基本信息查看
- 处理缺失值
- 处理重复值
- 数据保存

In [2]:
# 导入必要的库
import pandas as pd
import numpy as np
import os

In [4]:
# 读取数据
data_path = './data_raw/panel_data.xlsx'
df = pd.read_excel(data_path)
print("数据读取完成")
print(f"数据形状: {df.shape}")
df.head()

数据读取完成
数据形状: (720, 6)


,城市,年度,财政收入(亿元),财政支出(亿元),住户存款余额(亿元),地区生产总值(亿元)
0,上海,2006,1576.07,1795.57,8730.00,10825.4
1,上海,2007,2074.48,2181.68,8745.22,13179.8
2,上海,2008,2358.75,2593.92,11464.15,14877.1
3,上海,2009,2540.30,2989.65,13707.32,16181.4
4,上海,2010,2873.58,3302.89,15650.24,18319.6


In [5]:
# 查看数据基本信息
print("数据信息:")
df.info()
print("\n描述性统计:")
df.describe()
print("\n缺失值统计:")
df.isnull().sum()

数据信息:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 720 entries, 0 to 719
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   城市          720 non-null    object 
 1   年度          720 non-null    int64  
 2   财政收入(亿元)    682 non-null    float64
 3   财政支出(亿元)    682 non-null    float64
 4   住户存款余额(亿元)  679 non-null    float64
 5   地区生产总值(亿元)  682 non-null    float64
dtypes: float64(4), int64(1), object(1)
memory usage: 33.9+ KB

描述性统计:

缺失值统计:


城市             0
年度             0
财政收入(亿元)      38
财政支出(亿元)      38
住户存款余额(亿元)    41
地区生产总值(亿元)    38
dtype: int64

In [8]:
# 检查2025年的数据
print("2025年的数据:")
df_2025 = df[df['年度'] == 2025]
print(df_2025)
print(f"2025年数据行数: {len(df_2025)}")
print("年度分布:")
print(df['年度'].value_counts().sort_index())

2025年的数据:
       城市    年度  财政收入(亿元)  财政支出(亿元)  住户存款余额(亿元)  地区生产总值(亿元)
19     上海  2025       NaN       NaN         NaN         NaN
39   乌鲁木齐  2025       NaN       NaN         NaN         NaN
59     兰州  2025       NaN       NaN         NaN         NaN
79     北京  2025       NaN       NaN         NaN         NaN
99     南京  2025       NaN       NaN         NaN         NaN
119    南宁  2025       NaN       NaN         NaN         NaN
139    南昌  2025       NaN       NaN         NaN         NaN
159    厦门  2025       NaN       NaN         NaN         NaN
179    合肥  2025       NaN       NaN         NaN         NaN
199  呼和浩特  2025       NaN       NaN         NaN         NaN
219   哈尔滨  2025       NaN       NaN         NaN         NaN
239    大连  2025       NaN       NaN         NaN         NaN
259    天津  2025       NaN       NaN         NaN         NaN
279    太原  2025       NaN       NaN         NaN         NaN
299    宁波  2025       NaN       NaN         NaN         NaN
319    广州  2025       NaN     

In [9]:
# 数据清洗
# 1. 检查重复值
print("重复行数量:", df.duplicated().sum())

# 2. 删除2025年的数据行（由于完全缺失）
df_clean = df[df['年度'] != 2025].copy()
print(f"删除2025年后，数据形状: {df_clean.shape}")

# 3. 处理剩余缺失值 - 使用均值填充
numeric_cols = ['财政收入(亿元)', '财政支出(亿元)', '住户存款余额(亿元)', '地区生产总值(亿元)']
for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mean())

print("填充缺失值后，缺失值统计:")
print(df_clean.isnull().sum())

# 4. 检查数据类型
print("\n数据类型:")
print(df_clean.dtypes)

# 5. 确保年度是整数
df_clean['年度'] = df_clean['年度'].astype(int)

重复行数量: 0
删除2025年后，数据形状: (684, 6)
填充缺失值后，缺失值统计:
城市            0
年度            0
财政收入(亿元)      0
财政支出(亿元)      0
住户存款余额(亿元)    0
地区生产总值(亿元)    0
dtype: int64

数据类型:
城市             object
年度              int64
财政收入(亿元)      float64
财政支出(亿元)      float64
住户存款余额(亿元)    float64
地区生产总值(亿元)    float64
dtype: object


In [12]:
# 保存清洗后的数据
output_path = './data_clean/cleaned_panel_data_v2.csv'
df_clean.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"清洗后的数据已保存到: {output_path}")
print(f"保存的数据形状: {df_clean.shape}")

清洗后的数据已保存到: ./data_clean/cleaned_panel_data_v2.csv
保存的数据形状: (684, 6)


## 二. 清洗房地产数据（先预处理删除无效信息，再以地区作为key合并之后清洗）
- 房地产开发投资额.xls
- 房地产施工面积.xls
- 商品房销售额.xls
- 商品房销售面积.xls


合并并清洗 data_raw 目录的四个房地产类指标表：
- 房地产开发投资额.xls
- 房地产施工面积.xls
- 商品房销售额.xls
- 商品房销售面积.xls

预处理：
1) 去除表头三行（header=3，使第4行作为表头）
2) 去除表尾最后一行（skipfooter=1）
3) 去除“拉萨”所在行（地区=拉萨或包含拉萨）
4) 去除 2025 年数据

清洗与合并：
- 自动识别“地区列”和“年份列”；若检测到横向“年度列”则自动拉长
- 标准化为长表（地区, 年份, 指标, 值），并进一步透视为宽表
- 输出到 data_clean 目录（同时生成长表和宽表 CSV）

注意：若你的源表结构与常见统计表差异较大，可在函数中调整“地区列”和“年份列”的识别逻辑。

In [ ]:
import os
import re
import warnings
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd

# -----------------------
# 全局参数
# -----------------------
RAW_DIR = "data_raw"
CLEAN_DIR = "data_clean"

FILE_MAP = {
    "房地产开发投资额.xls": "房地产开发投资额",
    "房地产施工面积.xls": "房地产施工面积",
    "商品房销售额.xls": "商品房销售额",
    "商品房销售面积.xls": "商品房销售面积",
}

# 年份识别正则（列名或文本中的年份）
YEAR_NAME_RE = re.compile(r"(\d{4})\s*年?")
YEAR_VALUE_RE = re.compile(r"(^|\D)(\d{4})(\D|$)")

TARGET_EXCLUDE_YEAR = 2025
TARGET_EXCLUDE_REGION = "拉萨"


# -----------------------
# 工具函数
# -----------------------
def _choose_engine(file_path: str) -> str:
    """根据扩展名选择 pandas 读表引擎。"""
    ext = os.path.splitext(file_path.lower())[1]
    if ext == ".xls":
        return "xlrd"         # 旧版 Excel
    elif ext in (".xlsx", ".xlsm"):
        return "openpyxl"     # 新版 Excel
    else:
        # 默认尝试 openpyxl
        return "openpyxl"


def read_excel_smart(file_path: str) -> pd.DataFrame:
    """
    读取 Excel，按要求去除表头三行、表尾最后一行：
    - header=3  => 第 4 行作为列名
    - skipfooter=1 => 去除尾部合计等
    """
    engine = _choose_engine(file_path)
    try:
        df = pd.read_excel(
            file_path,
            engine=engine,
            header=3,      # 跳过前3行，第四行作为表头
            skipfooter=1,  # 去掉最后一行
            dtype=str      # 先全部以字符串读入，便于统一清洗
        )
    except Exception as e:
        raise RuntimeError(f"读取文件失败：{file_path}，请确认安装了必要引擎（xlrd/openpyxl）。原始错误：{e}")
    return df


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """去除列名首尾空白、全角空格、换行等；消除重复列名。"""
    def _norm(col: str) -> str:
        if col is None:
            return ""
        col = str(col)
        col = col.replace("\n", "").replace("\r", "")
        col = col.replace("\u3000", " ")  # 全角空格
        col = col.strip()
        return col

    df = df.copy()
    df.columns = [_norm(c) for c in df.columns]

    # 若存在重复列名，追加后缀
    counts = {}
    new_cols = []
    for c in df.columns:
        if c not in counts:
            counts[c] = 0
            new_cols.append(c)
        else:
            counts[c] += 1
            new_cols.append(f"{c}.{counts[c]}")
    df.columns = new_cols
    return df


def clean_string(s):
    """标准化单元格字符串：去空白、去全角空格，统一缺失值。"""
    if pd.isna(s):
        return np.nan
    s = str(s)
    s = s.replace("\u3000", " ").strip()
    if s in ["", "—", "-", "–", "— —", "——", "…", ".", "..", "...", "NA", "N/A", "nan", "Null", "NULL"]:
        return np.nan
    return s


def to_numeric_or_nan(x):
    """
    将数值类文本转换为 float：
    - 处理千分位逗号、空格、中文逗号
    - 处理百分号（如有则转换为小数：50% -> 0.5）
    - 其它无法解析的返回 NaN
    """
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x).strip()
    if s == "":
        return np.nan
    # 百分号处理
    if s.endswith("%"):
        s2 = s[:-1].replace(",", "").replace("，", "").replace(" ", "")
        try:
            return float(s2) / 100.0
        except:
            return np.nan
    # 常规数值
    s2 = s.replace(",", "").replace("，", "").replace(" ", "")
    # 有时会出现中文单位或注释，尝试取前导数值
    m = re.search(r"[-+]?\d+(\.\d+)?", s2)
    if m:
        try:
            return float(m.group())
        except:
            return np.nan
    return np.nan


def infer_region_column(df: pd.DataFrame) -> Optional[str]:
    """尝试识别地区列名。优先匹配典型命名，其次通过包含“拉萨”判断。"""
    candidates = [
        "地区", "地区(市)", "地区（市）", "地区名称", "地区/市", "地区-市",
        "省份", "省", "城市", "地市", "行政区", "地区（单位）", "单位"
    ]
    for c in df.columns:
        if any(key in c for key in candidates):
            return c

    # 回退策略：找一个文本列，且含有“拉萨”或省市名特征
    for c in df.columns:
        s = df[c].astype(str).fillna("")
        if s.str.contains(TARGET_EXCLUDE_REGION).any():
            return c

    # 再次回退：选择第一个 object 列
    for c in df.columns:
        if df[c].dtype == object:
            return c

    return None


def infer_year_columns_from_headers(df: pd.DataFrame) -> List[str]:
    """
    若存在横向年度列（如“2019年”、“2020年累计”等），返回这些列名；
    若返回空列表，则可能是纵向“年份”列。
    """
    year_cols = []
    for c in df.columns:
        m = YEAR_NAME_RE.search(str(c))
        if m:
            year = int(m.group(1))
            if 1990 <= year <= 2100:
                year_cols.append(c)
    return year_cols


def extract_year_series_from_column(series: pd.Series) -> pd.Series:
    """从一个列中提取年份（四位数），不可解析的为 NaN。"""
    def _get_year(v):
        if pd.isna(v):
            return np.nan
        # 数字或可转数字
        if isinstance(v, (int, float, np.number)):
            y = int(v)
            return y if 1990 <= y <= 2100 else np.nan
        s = str(v)
        m = YEAR_VALUE_RE.search(s)
        if m:
            y = int(m.group(2))
            if 1990 <= y <= 2100:
                return y
        return np.nan

    return series.apply(_get_year)


def infer_year_column_long(df: pd.DataFrame) -> Optional[str]:
    """
    若是纵向年份列，尝试在列名中查找['年','年份','时间','日期','period','date','月份']等关键字；
    找到后验证其可解析为年份的密度，返回最可能的年份列名。
    """
    year_col_candidates = [c for c in df.columns if any(k in c.lower() for k in ["年", "年份", "时间", "日期", "period", "date", "月份"])]
    best_col = None
    best_valid = -1
    for c in year_col_candidates:
        years = extract_year_series_from_column(df[c])
        valid = years.notna().sum()
        if valid > best_valid and valid >= max(1, int(0.2 * len(df))):  # 至少能识别 20% 行
            best_col = c
            best_valid = valid
    return best_col


def tidy_one_file(file_path: str, indicator_name: str) -> pd.DataFrame:
    """
    读取、预处理并清洗一个 Excel 文件，输出统一的长表：
    列：['地区', '年份', '指标', '值']
    """
    print(f"处理文件：{file_path}（指标：{indicator_name}）")
    df = read_excel_smart(file_path)
    df = normalize_columns(df)

    # 清理单元格空白等
    for c in df.columns:
        df[c] = df[c].map(clean_string)

    # 识别地区列
    region_col = infer_region_column(df)
    if region_col is None:
        warnings.warn(f"未能识别地区列，文件：{file_path}。将创建占位列 '地区'='未知'。")
        df["地区"] = "未知"
        region_col = "地区"

    # 丢弃空行（地区与其他关键列全空）
    df = df.dropna(how="all").copy()

    # 先过滤掉“拉萨”所在行（广义：包含“拉萨”）
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        df = df[~df[region_col].astype(str).str.contains(TARGET_EXCLUDE_REGION, na=False)]

    # 判断是横向年列(multi-year columns)还是纵向年份列
    year_header_cols = infer_year_columns_from_headers(df)

    tidy = None

    if len(year_header_cols) >= 1:
        # 横向年列：将这些列 melt 成长表
        id_vars = [c for c in df.columns if c not in year_header_cols]
        long_df = df.melt(id_vars=id_vars, value_vars=year_header_cols, var_name="年份列名", value_name="值")
        # 从“年份列名”中提取年份
        long_df["年份"] = long_df["年份列名"].apply(lambda x: int(YEAR_NAME_RE.search(str(x)).group(1)) if YEAR_NAME_RE.search(str(x)) else np.nan)
        long_df = long_df.drop(columns=["年份列名"])
        long_df = long_df.dropna(subset=["年份"])

        # 去除 2025 年
        long_df = long_df[long_df["年份"] != TARGET_EXCLUDE_YEAR]

        # 整理列
        long_df.rename(columns={region_col: "地区"}, inplace=True)
        # 数值清洗
        long_df["值"] = long_df["值"].map(to_numeric_or_nan)

        # 形成标准长表
        tidy = long_df[["地区", "年份", "值"]].copy()
        tidy["指标"] = indicator_name

    else:
        # 纵向年份列：尝试识别年份列
        year_col = infer_year_column_long(df)
        if year_col is None:
            warnings.warn(f"未能识别年份列，文件：{file_path}。将跳过该文件。")
            return pd.DataFrame(columns=["地区", "年份", "指标", "值"])

        # 提取年份
        df["年份"] = extract_year_series_from_column(df[year_col])
        df = df.dropna(subset=["年份"])
        df["年份"] = df["年份"].astype(int)

        # 去除 2025 年
        df = df[df["年份"] != TARGET_EXCLUDE_YEAR]

        # 可能存在多个数值列，这里尝试：
        # 1) 若只有一个明显数值列 => 直接用它
        # 2) 若存在“本年累计/当期值/合计”等优先列名 => 优先选择
        # 3) 否则尝试选择数值密度最高的一列
        value_candidates = [c for c in df.columns if c not in ["年份", region_col] and c not in FILE_MAP.values()]
        # 去掉明显的描述列
        value_candidates = [c for c in value_candidates if not any(k in c for k in ["单位", "指标", "注释", "说明"])]

        def score_numeric_density(series: pd.Series) -> int:
            vals = series.map(to_numeric_or_nan)
            return vals.notna().sum()

        # 优先关键字
        priority = ["本年累计", "累计", "当期值", "月度", "合计", "数值", "总量", "总计", "金额", "面积"]
        prioritized = [c for c in value_candidates if any(k in c for k in priority)]
        chosen_col = None

        if len(prioritized) == 1:
            chosen_col = prioritized[0]
        elif len(prioritized) > 1:
            chosen_col = max(prioritized, key=lambda c: score_numeric_density(df[c]))
        else:
            # 无优先关键字，找数值密度最高列
            if value_candidates:
                chosen_col = max(value_candidates, key=lambda c: score_numeric_density(df[c]))

        if chosen_col is None:
            warnings.warn(f"未能识别数值列，文件：{file_path}。将跳过该文件。")
            return pd.DataFrame(columns=["地区", "年份", "指标", "值"])

        # 数值清洗
        df["值"] = df[chosen_col].map(to_numeric_or_nan)
        df.rename(columns={region_col: "地区"}, inplace=True)

        tidy = df[["地区", "年份", "值"]].copy()
        tidy["指标"] = indicator_name

    # 去掉地区为空的行
    tidy["地区"] = tidy["地区"].map(clean_string)
    tidy = tidy.dropna(subset=["地区"])

    # 若地区列中带有层级（如“西藏自治区-拉萨市”），在上游已剔除含“拉萨”的行，这里再做一次保险过滤
    tidy = tidy[~tidy["地区"].astype(str).str.contains(TARGET_EXCLUDE_REGION, na=False)]

    # 值保留为数值
    tidy["值"] = tidy["值"].astype(float)

    # 最终列顺序
    tidy = tidy[["地区", "年份", "指标", "值"]].reset_index(drop=True)
    return tidy


def main():
    os.makedirs(CLEAN_DIR, exist_ok=True)

    all_tidy = []
    for fname, indicator in FILE_MAP.items():
        fpath = os.path.join(RAW_DIR, fname)
        if not os.path.exists(fpath):
            warnings.warn(f"未找到文件：{fpath}，将跳过。")
            continue
        td = tidy_one_file(fpath, indicator)
        if not td.empty:
            all_tidy.append(td)

    if not all_tidy:
        raise RuntimeError("未成功读取到任何数据，请检查 data_raw 目录与源文件结构。")

    # 合并全部长表
    long_df = pd.concat(all_tidy, ignore_index=True)

    # 再次保险：过滤 2025 和 拉萨
    long_df = long_df[(long_df["年份"] != TARGET_EXCLUDE_YEAR)]
    long_df = long_df[~long_df["地区"].astype(str).str.contains(TARGET_EXCLUDE_REGION, na=False)]

    # 宽表：按（地区, 年份）透视出各指标
    wide_df = long_df.pivot_table(
        index=["地区", "年份"],
        columns="指标",
        values="值",
        aggfunc="sum",   # 若同一（地区, 年份, 指标）多条记录，则求和（可按需改为 'first'）
        fill_value=np.nan
    ).reset_index()

    # 列扁平化（去掉列索引名）
    wide_df.columns.name = None

    # 输出 CSV（带 BOM，便于 Excel 打开）
    long_out = os.path.join(CLEAN_DIR, "cleaned_real_estate_long.csv")
    wide_out = os.path.join(CLEAN_DIR, "cleaned_real_estate_merged.csv")

    long_df.to_csv(long_out, index=False, encoding="utf-8-sig")
    wide_df.to_csv(wide_out, index=False, encoding="utf-8-sig")

    print(f"✅ 清洗后的长表已保存：{long_out}  （列：地区, 年份, 指标, 值）")
    print(f"✅ 合并后的宽表已保存：{wide_out}  （按地区+年份行，指标为列）")


if __name__ == "__main__":
    main()


处理文件：data_raw\房地产开发投资额.xls（指标：房地产开发投资额）
处理文件：data_raw\房地产施工面积.xls（指标：房地产施工面积）
处理文件：data_raw\商品房销售额.xls（指标：商品房销售额）
处理文件：data_raw\商品房销售面积.xls（指标：商品房销售面积）
✅ 清洗后的长表已保存：data_clean\real_estate_clean_long.csv  （列：地区, 年份, 指标, 值）
✅ 合并后的宽表已保存：data_clean\real_estate_clean_merged.csv  （按地区+年份行，指标为列）
